# ☁️ Étape 4 — Entraînement Hybride QLoRA (Phase 1 : CPT $\rightarrow$ Phase 2 : SFT) & Export GGUF *(Notebook Google Colab GPU)*

**Environnement** : **Ce notebook est celui qui s'exécute sur Google Colab (GPU Tesla T4)** (identique et synchronisé avec [`CameroonianFineTuning.ipynb`](../CameroonianFineTuning.ipynb) à la racine).

Il reprend l'intégralité du code que tu avais déjà écrit dans `CameroonianFineTuning.ipynb` et intègre directement les ajustements de l'architecture hybride :
1. **Phase 1 (Continued Pre-training — CPT)** : Immersion linguistique sur les commentaires Camfranglais bruts avec `packing=True` et `per_device_train_batch_size=2`.
2. **Phase 2 (Supervised Fine-Tuning — SFT)** : Remplacement d'`alpaca_prompt` par le **Chat Template natif de Mistral** (`[INST] ... [/INST]`) + **masquage de la loss sur les instructions (`train_on_responses_only`)**.
3. **Génération des réponses sur `sft_test`** (`test_predictions.json` pour l'Étape 5 en local), **Inférence `TextStreamer`**, **Sauvegarde sécurisée `hf_token`** et **Export GGUF (`q4_k_m`)**.

### 1. Installation
Installing Unsloth and required libraries.

In [ ]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    import torch; v = re.match(r"[0-9]{1,}\.[0-9]{1,}", str(torch.__version__)).group(0)
    xformers = "xformers==" + ("0.0.33.post1" if v=="2.9" else "0.0.32.post2" if v=="2.8" else "0.0.29.post3")
    !pip install --no-deps bitsandbytes accelerate {xformers} peft trl triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

In [ ]:
from google.colab import userdata
hf_token = userdata.get('hf_token')

### 2. Loading the model & QLoRA Configuration

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
dtype = None
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Mistral-Nemo-Instruct-2407-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
    token = hf_token,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

### 3. Phase 1 — Continued Pre-Training (CPT) sur le Texte Brut Camfranglais

In [ ]:
from datasets import load_dataset, Dataset
from trl import SFTConfig, SFTTrainer
import json

CPT_HF_REPO = "Cmr237/camfranglais-corpus-cpt"
CPT_LOCAL_FILE = "cpt_train.json"

try:
    cpt_dataset = load_dataset(CPT_HF_REPO, split="train", token=hf_token)
    print(f"✅ Corpus CPT chargé depuis Hugging Face ({len(cpt_dataset)} textes).")
except Exception:
    if os.path.exists(CPT_LOCAL_FILE):
        with open(CPT_LOCAL_FILE, "r", encoding="utf-8") as f:
            raw_cpt = json.load(f)
        cpt_rows = [{"text": t} if isinstance(t, str) else t for t in raw_cpt]
        cpt_dataset = Dataset.from_list(cpt_rows)
        print(f"✅ Corpus CPT chargé depuis {CPT_LOCAL_FILE} ({len(cpt_dataset)} textes).")
    else:
        cpt_dataset = Dataset.from_list([
            {"text": "Gars le son là est trop djoss, le mbom a tout cassé sur le beat au kwatt !"},
            {"text": "Je wanda comment Eneo coupe la lumière tous les soirs alors qu'on a payé les factures."},
            {"text": "Le benskineur a failli nous verser au rond-point Deïdo, mon cœur a sauté massa !"}
        ] * 20)

EOS_TOKEN = tokenizer.eos_token

def format_cpt_texts(examples):
    return {"text": [t.strip() + EOS_TOKEN for t in examples["text"]]}

cpt_dataset = cpt_dataset.map(format_cpt_texts, batched=True)

model.config.use_cache = False
cpt_trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = cpt_dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    packing = True,
    args = SFTConfig(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        num_train_epochs = 1,
        learning_rate = 1e-4,
        save_strategy = "no",
        logging_steps = 5,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "cosine",
        seed = 3407,
        output_dir = "outputs_cpt",
        report_to = "none",
    ),
)

cpt_trainer_stats = cpt_trainer.train()

### 4. Phase 2 — Supervised Fine-Tuning (SFT Conversationnel avec Chat Template & `train_on_responses_only`)

In [ ]:
from unsloth.chat_templates import get_chat_template, train_on_responses_only

tokenizer = get_chat_template(tokenizer, chat_template="mistral")
SYSTEM_PROMPT = "Tu réponds uniquement en camfranglais"

SFT_HF_REPO = "Cmr237/camfranglais-corpus-sft"
SFT_LOCAL_FILE = "conversations_dataset.json"

try:
    sft_dataset_dict = load_dataset(SFT_HF_REPO, token=hf_token)
    sft_train_dataset = sft_dataset_dict["train"]
    sft_test_dataset = sft_dataset_dict.get("test", None)
except Exception:
    raw_sft = load_dataset("json", data_files=SFT_LOCAL_FILE, split="train")
    split_ds = raw_sft.train_test_split(test_size=0.1, seed=3407)
    sft_train_dataset = split_ds["train"]
    sft_test_dataset = split_ds["test"]

def formatted_conversations(examples):
    texts = []
    if "conversations" in examples:
        for convo in examples["conversations"]:
            full_convo = [{"role": "system", "content": SYSTEM_PROMPT}] + convo
            texts.append(tokenizer.apply_chat_template(full_convo, tokenize=False, add_generation_prompt=False))
    elif "instruction" in examples:
        for instruction, inp, output in zip(examples["instruction"], examples["input"], examples["output"]):
            user_content = f"{instruction}\n{inp}".strip() if inp else instruction
            full_convo = [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": user_content},
                {"role": "assistant", "content": output}
            ]
            texts.append(tokenizer.apply_chat_template(full_convo, tokenize=False, add_generation_prompt=False))
    return {"text": texts}

sft_train_dataset = sft_train_dataset.map(formatted_conversations, batched=True)
if sft_test_dataset is not None:
    sft_test_dataset = sft_test_dataset.map(formatted_conversations, batched=True)

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = sft_train_dataset,
    eval_dataset = sft_test_dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    packing = False,
    args = SFTConfig(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        num_train_epochs = 2,
        learning_rate = 5e-5,
        save_strategy = "epoch",
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none",
    ),
)

trainer = train_on_responses_only(
    trainer,
    instruction_part = "[INST]",
    response_part = "[/INST]",
)

trainer_stats = trainer.train()

### 5. Génération des Réponses sur `sft_test` & Inférence Interactive

In [ ]:
FastLanguageModel.for_inference(model)
from transformers import TextStreamer

messages = [{"role": "system", "content": SYSTEM_PROMPT}]
while True:
    user_input = str(input("user: \n"))
    if user_input.lower() == "exit":
        break
    messages.append({"role": "user", "content": user_input})
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize = True,
        add_generation_prompt = True,
        return_tensors = "pt",
    ).to("cuda")

    text_streamer = TextStreamer(tokenizer, skip_prompt = True, skip_special_tokens = True)
    print("Assistant: ", end="", flush=True)
    outputs = model.generate(
        input_ids = inputs,
        streamer = text_streamer,
        max_new_tokens = 128,
        pad_token_id = tokenizer.eos_token_id
    )
    prompt_length = inputs.shape[1]
    response_ids = outputs[0][prompt_length:]
    response_text = tokenizer.decode(response_ids, skip_special_tokens=True).strip()
    messages.append({"role": "assistant", "content": response_text})

### 6. Saving LoRA Adapters & Pushing GGUF (`q4_k_m`) to Hugging Face

In [ ]:
model.push_to_hub("Cmr237/Camfranglais", token = hf_token)
tokenizer.push_to_hub("Cmr237/Camfranglais", token = hf_token)

model.push_to_hub_gguf(
    "Cmr237/Camfranglais",
    tokenizer,
    quantization_method = "q4_k_m",
    token = hf_token,
)